## settings

In [ ]:
TOP_N_CAST = 5 # Number of top cast members to extract
TOP_N_GENRES = 5 # Number of top genres to extract¨


#data paths
name_of_orginel_folder_data= "All_data"
save_data_path="../data/"

#
number_of_rows_to_load = None # Set to None to load all rows, or specify a number for testing
keep_colums_list=['budget',
 'genres',
 'id',
 'popularity',
 'release_date',
 'revenue',
 'runtime',
 'vote_average',
 'vote_count']

## load Data and pakages

In [ ]:
from pathlib import Path
import pandas as pd
import ast

# Resolve the Data folder relative to this notebook location.
data_dir = (Path.cwd().parent / name_of_orginel_folder_data).resolve()
csv_files = sorted(data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")
# Load the first number_of_rows_to_load from each CSV into a dictionary keyed by filename.
if number_of_rows_to_load is None:
    datasets = {csv_file.stem: pd.read_csv(csv_file) for csv_file in csv_files}
else:
    datasets = {csv_file.stem: pd.read_csv(csv_file, nrows=number_of_rows_to_load) for csv_file in csv_files}

print(f"Loaded {len(datasets)} datasets from {data_dir}")
for name, df in datasets.items():
    print(f"- {name}: {df.shape[0]} rows, {df.shape[1]} columns")

C:\Users\mikke\AppData\Local\Temp\ipykernel_2184\3373344098.py:12: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  datasets = {csv_file.stem: pd.read_csv(csv_file) for csv_file in csv_files}


Loaded 7 datasets from C:\Users\mikke\Documents\Model-based-machine-learning\Data
- credits: 45476 rows, 3 columns
- keywords: 46419 rows, 2 columns
- links: 45843 rows, 3 columns
- links_small: 9125 rows, 3 columns
- movies_metadata: 45466 rows, 24 columns
- ratings: 26024289 rows, 4 columns
- ratings_small: 100004 rows, 4 columns


In [3]:
credits=datasets['credits'].copy()
keywords=datasets['keywords']
links=datasets['links']
links_small=datasets['links_small']
movies_metadata=datasets['movies_metadata']
ratings=datasets['ratings']
ratings_small=datasets['ratings_small']

## data work

In [4]:
# Convert 'cast' and 'crew' columns from string representation of lists/dictionaries to actual lists/dictionaries
credits['cast'] = credits['cast'].apply(ast.literal_eval)
credits['crew'] = credits['crew'].apply(ast.literal_eval)

In [6]:
# Build a unique actor table from the existing credits DataFrame.
actor_rows = []

for _, row in credits.iterrows():
    cast_list = row['cast'] if isinstance(row['cast'], list) else []

    for member in cast_list:
        actor_rows.append({
            'actor_id': int(member['id']),
            'actor_name': member['name']
        })

actor_df = pd.DataFrame(actor_rows).drop_duplicates(subset=['actor_id']).reset_index(drop=True)

print(f"Created actor table with {actor_df.shape[0]} unique actors")


Created actor table with 206158 unique actors


In [7]:
def extract_top_cast_ids(df, order_number) -> pd.DataFrame:
    def get_top_ids(cast_list):
        if not isinstance(cast_list, list):
            return [None] * order_number
        
        cast_sorted = sorted(cast_list, key=lambda x: x['order'])
        top_ids = [int(member['id']) for member in cast_sorted[:order_number]]
        top_ids += [None] * (order_number - len(top_ids))
        
        return top_ids

    top_ids_series = df['cast'].apply(get_top_ids)
    
    for i in range(order_number):
        df[f'Actor_{i+1}'] = top_ids_series.apply(lambda x: x[i])
    df.pop('cast')
    df.pop('crew')
    return df

In [8]:
credits = extract_top_cast_ids(credits, TOP_N_CAST)

In [9]:
credits.head(2)

,id,Actor_1,Actor_2,Actor_3,Actor_4,Actor_5
0,862,31.0,12898.0,7167.0,12899.0,12900.0
1,8844,2157.0,8537.0,205.0,145151.0,5149.0


In [10]:
# credits.to_csv( "../Generated_data/credits_processed.csv", index=False)
# actor_df.to_csv( "../Generated_data/actor_table.csv", index=False)

## movies_metadata

In [11]:
movies_metadata['genres'] = movies_metadata['genres'].apply(ast.literal_eval)


In [12]:
def create_genre_counts_df(movies_metadata: pd.DataFrame) -> pd.DataFrame:
    genre_counts = {}

    for _, row in movies_metadata.iterrows():
        genre_value = row['genres']

        if isinstance(genre_value, str):
            try:
                genre_list = ast.literal_eval(genre_value)
            except (ValueError, SyntaxError):
                genre_list = []
        elif isinstance(genre_value, list):
            genre_list = genre_value
        else:
            genre_list = []

        for member in genre_list:
            genre_id = int(member['id'])
            genre_name = member['name']

            if genre_id not in genre_counts:
                genre_counts[genre_id] = {'genre_name': genre_name, 'count_of_genre': 0}

            genre_counts[genre_id]['count_of_genre'] += 1

    genre_df = (
        pd.DataFrame.from_dict(genre_counts, orient='index')
        .reset_index()
        .rename(columns={'index': 'genre_id'})
        .loc[:, ['genre_id', 'genre_name', 'count_of_genre']]
        .sort_values(['count_of_genre', 'genre_name'], ascending=[False, True])
        .reset_index(drop=True)
    )

    return genre_df


# Build a genre table that includes how many movies each genre appears in.
genre_df2 = create_genre_counts_df(movies_metadata)
#print(f"Created genre table with {genre_df.shape[0]} unique genres")
genre_df=genre_df2[0:TOP_N_GENRES]


In [13]:
def keep_columns(column_names: list[str], df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, [col for col in column_names if col in df.columns]].copy()

In [14]:
movies_metadata=keep_columns(keep_colums_list,movies_metadata)

In [15]:
def expand_genres_to_columns(movies_metadata: pd.DataFrame, genre_df: pd.DataFrame) -> pd.DataFrame:
    movies_out = movies_metadata.copy()

    target_genres = {
        int(row['genre_id']): str(row['genre_name'])
        for _, row in genre_df[['genre_id', 'genre_name']].drop_duplicates().iterrows()
    }

    genre_column_map = {
        genre_id: f"genre_{genre_name.lower().replace(' ', '_')}"
        for genre_id, genre_name in target_genres.items()
    }

    for col_name in genre_column_map.values():
        movies_out[col_name] = 0.0

    other_genre_values = []

    for idx, value in movies_out['genres'].items():
        if isinstance(value, str):
            try:
                row_genres = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                row_genres = []
        elif isinstance(value, list):
            row_genres = value
        else:
            row_genres = []

        total_genres = max(len(row_genres), 1)
        weight = round(1 / total_genres, 2)
        
        found_known_genre = False

        for item in row_genres:
            genre_id = item.get('id') if isinstance(item, dict) else None
            if genre_id in genre_column_map:
                movies_out.at[idx, genre_column_map[genre_id]] = weight
                found_known_genre = True

        other_genre_values.append(0.0 if found_known_genre else weight)

    movies_out['other_genre'] = other_genre_values
    movies_out = movies_out.drop(columns=['genres'])

    return movies_out


movies_metadata_with_genres = expand_genres_to_columns(movies_metadata, genre_df)


In [16]:
# Ensure matching dtype for join key
movies_metadata_with_genres['id'] = pd.to_numeric(movies_metadata_with_genres['id'], errors='coerce')
credits['id'] = pd.to_numeric(credits['id'], errors='coerce')

movies_with_genres_and_cast = movies_metadata_with_genres.merge(credits, on='id', how='left')


In [ ]:
movies_with_genres_and_cast.to_csv( f"{save_data_path}movies_with_genres_and_cast.csv", index=False)
actor_df.to_csv( f"{save_data_path}actor_table.csv", index=False)